In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, get_json_object, coalesce, lit, when, cast
from pyspark.sql.types import DecimalType
from datetime import datetime

# Initialize Spark Session
spark = SparkSession.builder.appName("Ageing_Analysis").getOrCreate()

# Read source tables
invoice_schedules = spark.read.table("furlenco_silver.furbooks_evolve.Invoice_Schedules")
invoice_cycles = spark.read.table("furlenco_silver.furbooks_evolve.invoice_cycles")
invoices = spark.read.table("furlenco_silver.furbooks_evolve.Invoices")
items = spark.read.table("furlenco_silver.order_management_systems_evolve.items")
orders = spark.read.table("furlenco_silver.order_management_systems_evolve.Orders")
address_snapshots = spark.read.table("furlenco_silver.furbooks_evolve.Address_Snapshots")
payments = spark.read.table("furlenco_silver.gringotts_evolve.Payments")
credit_notes = spark.read.table("furlenco_silver.furbooks_evolve.Credit_notes")
outstanding_settlements = spark.read.table("furlenco_silver.furbooks_evolve.Outstanding_Settlements")

# Step 1: Create final CTE - Join multiple tables and extract JSON fields
final = (
    invoice_schedules.alias("ins")
    .join(
        invoice_cycles.alias("ic"),
        col("ic.invoice_schedule_id") == col("ins.id"),
        "left"
    )
    .join(
        invoices.alias("i"),
        col("i.id") == col("ic.invoice_id"),
        "left"
    )
    .join(
        items.alias("it"),
        (col("it.id") == col("ins.accountable_entity_id")) & 
        (col("ins.accountable_entity_type") == lit("ITEM")),
        "left"
    )
    .join(
        orders.alias("o"),
        col("it.order_id") == col("o.id"),
        "left"
    )
    .join(
        address_snapshots.alias("ads"),
        col("o.snapshotted_billing_address_id") == col("ads.reference_id"),
        "left"
    )
    .join(
        payments.alias("g_pay"),
        get_json_object(col("o.payment_details"), "$.id") == col("g_pay.id"),
        "left"
    )
    .select(
        col("ins.accountable_entity_id").alias("accountable_entity_id"),
        col("ins.accountable_entity_type").alias("accountable_entity_type"),
        col("it.name").alias("item_name"),
        col("i.customer_identifier").alias("fur_id"),
        col("i.serial_code").alias("invoice_number"),
        col("i.issue_date").alias("invoice_date"),
        get_json_object(col("ic.monetary_components"), "$.taxableAmount").alias("taxable_value"),
        get_json_object(col("ic.monetary_components"), "$.tax.breakup.cgst.amount").alias("CGST"),
        get_json_object(col("ic.monetary_components"), "$.tax.breakup.sgst.amount").alias("SGST"),
        get_json_object(col("ic.monetary_components"), "$.tax.breakup.igst.amount").alias("IGST"),
        get_json_object(col("ic.monetary_components"), "$.tax.breakup.cgst.rate").alias("cgst_rate"),
        get_json_object(col("ic.monetary_components"), "$.tax.breakup.sgst.rate").alias("sgst_rate"),
        get_json_object(col("ic.monetary_components"), "$.tax.breakup.igst.rate").alias("igst_rate"),
        col("ic.start_date").alias("cycle_start_date"),
        col("ic.end_date").alias("cycle_end_date"),
        col("i.vertical"),
        get_json_object(col("i.Data"), "$.city").alias("city_name"),
        col("i.id").alias("invoice_Id"),
        col("ads.gstin"),
        get_json_object(col("o.payment_details"), "$.id").alias("payment_id")
    )
    .filter(col("ic.state") == lit("INVOICED"))
)

# Step 2: Filter by date range
final_final = final.filter(
    (col("invoice_date") >= "2025-04-01") &
    (col("invoice_date") <= "2025-11-30")
)

# Step 3: Apply credit notes filter
settlement_credit_notes = outstanding_settlements.filter(
    col("source") != "WAIVER"
).select(col("credit_note_id")).distinct()

after_cn = (
    final_final.alias("ff")
    .join(
        credit_notes.alias("cre"),
        col("cre.invoice_id") == col("ff.invoice_Id"),
        "left"
    )
    .join(
        settlement_credit_notes,
        col("cre.id") == col("credit_note_id"),
        "left_semi"
    )
    .select(
        col("ff.*"),
        col("cre.id").alias("ignore_creds")
    )
)

# Step 4: Final aggregation and calculation
result = after_cn.filter(
    (col("ignore_creds").isNull()) &
    (col("vertical") == lit("FURLENCO_RENTAL"))
).select(
    col("fur_id").alias("identifier"),
    col("invoice_Id").alias("invoice_id"),
    col("invoice_date"),
    col("taxable_value"),
    col("CGST"),
    col("SGST"),
    col("IGST"),
    (
        coalesce(col("taxable_value").cast(DecimalType(18, 4)), lit(0)) +
        coalesce(col("CGST").cast(DecimalType(18, 4)), lit(0)) +
        coalesce(col("SGST").cast(DecimalType(18, 4)), lit(0)) +
        coalesce(col("IGST").cast(DecimalType(18, 4)), lit(0))
    ).alias("invoice_value")
)

# Display results
result.show(truncate=False)

# Optional: Save to table
# result.write.mode("overwrite").saveAsTable("furlenco_silver.finance.ageing_analysis")


